In [ ]:
import sys
sys.path.insert(0, '/home/phil/Dev/Python/Single Cell/scanpy')
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import tarfile
from urllib.request import urlopen, Request
url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
ua = "Mozilla/5.0 (Android; Mobile; rv:40.0) Gecko/40.0 Firefox/40.0"
req = Request(url, data=None, headers={'User-Agent': ua})
path_data = Path("data")
path_matrices = path_data / "filtered_gene_bc_matrices"
if not path_matrices.is_dir():
    with urlopen(req) as r:
        with tarfile.open(fileobj=r, mode='r|gz') as archive:
            archive.extractall(path_data)

In [ ]:
adata = sc.read_10x_mtx(path_matrices / "hg19", var_names='gene_symbols', cache=True)

# preprocess

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
mito_genes = adata.var_names.str.startswith('MT-')
adata.obs['percent_mito'] = np.sum(adata[:, mito_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1
adata.obs['n_counts'] = adata.X.sum(axis=1).A1

In [ ]:
adata = adata[adata.obs.n_genes < 2500, :]
adata = adata[adata.obs.percent_mito < 0.05, :]

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var['highly_variable']]

In [ ]:
sc.pp.regress_out(adata, ['n_counts', 'percent_mito'])

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# analyze

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata)

In [ ]:
marker_genes = [
    'IL7R', 'CD79A', 'MS4A1', 'CD8A', 'CD8B', 'LYZ',
    'CD14', 'LGALS3', 'S100A8', 'GNLY', 'NKG7', 'KLRB1',  
    'FCGR3A', 'MS4A7', 'FCER1A', 'CST3', 'PPBP',
]

In [ ]:
new_cluster_names = [
    'CD4 T',
    'CD14-\nMonocytes',
    'B',
    'CD8 T', 
    'NK',
    'FCGR3A-\nMonocytes',
    'Dendritic',
    'Mega-\nkaryocytes',
]
adata.rename_categories('leiden', new_cluster_names)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

In [ ]:
sc.settings.set_figure_params(fontsize=10)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.serif'] = ['Palatino', 'Palatino Linotype', 'TeX Gyre Pagella']
plt.rcParams['lines.linewidth'] = .5

fig = plt.figure(figsize=(7.6, 2))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1], wspace=0.05)
gssub = gs[0].subgridspec(1, 2, wspace=.55)
axs = [fig.add_subplot(gssub[0]), fig.add_subplot(gssub[1]), fig.add_subplot(gs[1])]

sc.pl.rank_genes_groups_violin(adata, groups=adata.obs['leiden'].cat.categories[0], n_genes=8, strip=False, ax=axs[0], show=False)
sc.pl.stacked_violin(adata, marker_genes, groupby='leiden', rotation=90,  ax=axs[1], show=False)
sc.pl.umap(adata, color='leiden', legend_loc='on data', legend_fontsize='x-small', title='', frameon=False, ax=axs[2], show=False)

axs[0].set_title('0 vs. rest')
axs[0].set_xlabel('Genes')
axs[0].set_ylabel('Expression')
fig.savefig('scanpy.pdf', bbox_inches='tight')